# smiles2iupac — Benchmarking notebook

Runs the conversion pipeline against every molecule in `tests/fixtures/known_molecules.csv` and reports:

- exact-match rate per layer (cache, PubChem, total miss)
- median + p95 latency overall and per layer
- failure breakdown by error type
- distribution of name sources as a bar chart

**Configuration:** `Pipeline(use_pubchem=True, use_stout=False)` — measures the core PubChem path only. STOUT/OPSIN are weekend-2 layers and not required to run this report.

**Live network:** PubChem is queried at ~5 req/s. With ~260 molecules expect roughly 60–90 seconds for a cold run. Re-runs hit the cache and complete in <1s.

In [ ]:
from __future__ import annotations

import csv
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
# matplotlib is not a hard dep of smiles2iupac; install it in your venv if missing:
#   /path/to/.venv/bin/python -m pip install matplotlib
# (or, with uv: VIRTUAL_ENV=/path/to/.venv uv pip install matplotlib)
import matplotlib.pyplot as plt

# Resolve repo root by walking up from CWD until we find the fixture file.
# This works whether the notebook is executed from notebooks/, the repo root,
# or via `jupyter nbconvert --execute` from anywhere.
def _find_repo_root(start: Path) -> Path:
    target = Path('tests/fixtures/known_molecules.csv')
    cur = start.resolve()
    for parent in [cur, *cur.parents]:
        if (parent / target).exists():
            return parent
    raise FileNotFoundError(
        f'Could not locate {target} above {start} — run from the smiles2iupac repo'
    )

REPO_ROOT = _find_repo_root(Path.cwd())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

FIXTURE = REPO_ROOT / 'tests' / 'fixtures' / 'known_molecules.csv'
print(f'Repo root: {REPO_ROOT}')
print(f'Fixture:   {FIXTURE}')

In [ ]:
from smiles2iupac.pipeline import Pipeline
from smiles2iupac.cache import Cache
from smiles2iupac.result import Source

# Use a fresh, in-memory cache so we measure a true cold pass through the
# stack. Set use_pubchem=True, use_stout=False for the core measurement.
TMP_CACHE_PATH = REPO_ROOT / '.benchmark_cache.db'
if TMP_CACHE_PATH.exists():
    TMP_CACHE_PATH.unlink()
cache = Cache(TMP_CACHE_PATH)
pipeline = Pipeline(cache=cache, use_pubchem=True, use_stout=False)

## Load fixture

Each row contains `name, smiles, canonical_smiles, note`. The benchmark only needs `name` (for the curated label) and `smiles` (the input).

In [ ]:
with FIXTURE.open() as f:
    rows = list(csv.DictReader(f))

print(f'Loaded {len(rows)} molecules')
note_counts = Counter(r['note'] or '(none)' for r in rows)
for note, count in note_counts.most_common():
    print(f'  {note:30s}  {count}')

## Run the pipeline

For each row we capture: name, source, confidence, latency in ms, and any error string. Latency is wall-clock around `pipeline.convert()`.

In [ ]:
records: list[dict] = []
t_total_start = time.perf_counter()
for i, row in enumerate(rows):
    smiles = row['smiles']
    expected_name = row['name']
    note = row['note']

    t0 = time.perf_counter()
    try:
        result = pipeline.convert(smiles)
        latency_ms = (time.perf_counter() - t0) * 1000.0
        rec = {
            'expected_name': expected_name,
            'smiles': smiles,
            'note': note,
            'returned_name': result.name,
            'source': result.source.value if result.source else None,
            'confidence': result.confidence,
            'kind': result.kind,
            'latency_ms': latency_ms,
            'error': result.error,
            'warnings': '; '.join(result.warnings) if result.warnings else None,
        }
    except Exception as exc:  # pragma: no cover — surface in report
        latency_ms = (time.perf_counter() - t0) * 1000.0
        rec = {
            'expected_name': expected_name,
            'smiles': smiles,
            'note': note,
            'returned_name': None,
            'source': None,
            'confidence': 0.0,
            'kind': None,
            'latency_ms': latency_ms,
            'error': f'EXCEPTION: {type(exc).__name__}: {exc}',
            'warnings': None,
        }
    records.append(rec)
    if (i + 1) % 25 == 0:
        print(f'  {i + 1}/{len(rows)} done')

elapsed = time.perf_counter() - t_total_start
print(f'\nFinished {len(records)} conversions in {elapsed:.1f}s')

## Source distribution & hit rates

Every successful conversion comes from one of the source layers. Misses have `source=None` (or `'none'`).

In [ ]:
source_counter = Counter(
    (r['source'] or 'none') if r['returned_name'] else 'miss'
    for r in records
)
total = len(records)
print(f'Total molecules: {total}\n')
print(f'{"source":25s}  {"count":>6s}  {"%":>6s}')
for source, count in sorted(source_counter.items(), key=lambda kv: -kv[1]):
    pct = 100.0 * count / total
    print(f'{source:25s}  {count:>6d}  {pct:>5.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = list(source_counter.keys())
values = [source_counter[k] for k in labels]
colors = ['#2a9d8f' if k == 'cache' else '#264653' if k == 'pubchem' else '#e76f51' for k in labels]
ax.bar(labels, values, color=colors)
ax.set_ylabel('count')
ax.set_title('Name source distribution')
for i, v in enumerate(values):
    ax.text(i, v + total * 0.01, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## Latency: overall and per layer

Median is the typical case; p95 catches the slow tail (network jitter, large molecules, retries). Cache lookups should be sub-millisecond; PubChem hits are dominated by network round-trip (~100–400 ms typical from a US laptop).

In [ ]:
all_latencies = np.array([r['latency_ms'] for r in records])
by_source: dict[str, list[float]] = defaultdict(list)
for r in records:
    key = (r['source'] or 'none') if r['returned_name'] else 'miss'
    by_source[key].append(r['latency_ms'])

def fmt_stats(arr: np.ndarray) -> str:
    if arr.size == 0:
        return '(no samples)'
    return (
        f'n={arr.size:>4d}  '
        f'median={np.median(arr):>7.1f} ms  '
        f'p95={np.percentile(arr, 95):>7.1f} ms  '
        f'mean={arr.mean():>7.1f} ms  '
        f'max={arr.max():>7.1f} ms'
    )

print('Overall:')
print(f'  {fmt_stats(all_latencies)}\n')
print('Per source:')
for source, latencies in sorted(by_source.items()):
    arr = np.array(latencies)
    print(f'  {source:12s}  {fmt_stats(arr)}')

## Failure breakdown

Molecules that came back without a name — typically PubChem misses, classifier rejections (reactions/polymers), or transient HTTP errors.

In [ ]:
failures = [r for r in records if not r['returned_name']]
print(f'{len(failures)} failures out of {len(records)} ({100*len(failures)/len(records):.1f}%)\n')

# Bucket errors by leading prefix (first colon-delimited segment)
def error_bucket(err: str | None) -> str:
    if not err:
        return '(no error message)'
    # 'pubchem unavailable: ...' -> 'pubchem unavailable'
    head = err.split(':', 1)[0].strip()
    return head if head else '(empty)'

buckets = Counter(error_bucket(r['error']) for r in failures)
print(f'{"error class":40s}  {"count":>6s}')
for label, count in buckets.most_common():
    print(f'{label:40s}  {count:>6d}')

In [ ]:
# Show a sample of failures with their original SMILES — useful for spot-debugging
print('Sample of up to 15 failures:\n')
for r in failures[:15]:
    print(f'  {r["expected_name"]:30s}  [{r["smiles"]:40s}]')
    print(f'    -> {r["error"]}')
    if r['warnings']:
        print(f'    !  {r["warnings"]}')

## Cache replay (re-run with warmed cache)

Every successful PubChem hit gets stored to the cache. Re-running with the same cache should drop almost all latency to <1 ms per call.

In [ ]:
t0 = time.perf_counter()
replay = []
for r in records:
    s0 = time.perf_counter()
    res = pipeline.convert(r['smiles'])
    replay.append(
        {
            'source': res.source.value if res.source else None,
            'latency_ms': (time.perf_counter() - s0) * 1000.0,
            'name': res.name,
        }
    )
elapsed = time.perf_counter() - t0

lat = np.array([x['latency_ms'] for x in replay])
src = Counter(x['source'] or 'none' for x in replay)
print(f'Replay: {len(replay)} calls in {elapsed:.2f}s')
print(f'  median={np.median(lat):.2f} ms  p95={np.percentile(lat, 95):.2f} ms')
print('  source distribution:')
for k, v in src.most_common():
    print(f'    {k:15s}  {v}')

In [ ]:
# Cleanup the temp cache file so subsequent runs start cold
if TMP_CACHE_PATH.exists():
    TMP_CACHE_PATH.unlink()
    print(f'Removed {TMP_CACHE_PATH.name}')

## Summary

What this notebook measures:

- **Coverage:** how many fixture molecules can the pipeline name?
  Successful PubChem hits are the floor for a useful tool; misses point to molecules that need STOUT (weekend 2) or human curation.
- **Latency:** PubChem dominates the tail. Cache layer should be sub-millisecond.
  If p95 PubChem latency exceeds ~1 s, suspect rate-limiting or network.
- **Failure mix:** classifier rejections (salts, reactions, polymers) vs PubChem   misses vs network errors. The first two are by design; the third is an alert.

**Caveats:**

1. PubChem is queried live — results depend on network and PubChem uptime.
2. The fixture is canonicalization-grade, not naming-grade: many `name` values    are common names that PubChem may return as synonyms rather than primary IUPAC.
3. STOUT is intentionally disabled here. Re-run with `use_stout=True` to extend    coverage onto miss molecules once STOUT deps are installed.
